# Sklepanje v negotovosti

V dosedanjih predavanjih smo obravnavali predvsem deterministične modele predstavitve znanja in sklepanja. Pri takih modelih predpostavimo, da so dejstva o svetu bodisi resnična bodisi neresnična, pravila pa veljajo brez izjem. Tak pristop je uporaben, kadar je znanje popolno in natančno. V resničnih problemih pa pogosto nimamo popolnih informacij: opazovanja so lahko nepopolna, meritve nenatančne, mehanizmi vzroka in posledice pa niso povsem zanesljivi in lahko ocenimo zgolj njihovo verjetnost v podani situaciji.

Razmislimo o enostavnem primeru vžiga avtomobila. Opazujemo tri boolove spremenljivke:

- $\textit{Gorivo}, G$, ki ima vrednosti $\textit{je}$ in $\textit{ni}$,
- $\textit{ČisteSvečke}, S$, ki ima vrednosti $\textit{da}$ in $\textit{ne}$, ter
- $\textit{VžigAvtomobila}, V$, ki ima vrednosti $\textit{da}$ in $\textit{ne}$.

V determinističnem modelu bi lahko zapisali pravila:

1. Če v avtomobilu ni goriva, potem avtomobil ne vžge.
1. Če svečke niso čiste, potem avtomobil ne vžge.
1. Če so svečke čiste in je v avtomobilu gorivo, potem avtomobil vžge.

Tak opis je uporaben, vendar hitro naleti na omejitve. Če opazimo, da velja `V=ne`, bi radi vedeli, kateri vzrok je verjetnejši. Ali je razlog bolj verjetno pomanjkanje goriva ali umazanost svečk? Kaj pa če opazimo, da je v avtomobilu gorivo in so svečke čiste, avtomobil pa še vedno ne vžge? Deterministična pravila v takem primeru pogosto ne zadostujejo. Potrebujemo model, ki ne zna opisati le možnih situacij, temveč tudi njihovo verjetnost.

To je osnovna motivacija za sklepanje v negotovosti. V teh zapiskih bomo uvedli osnovne verjetnostne pojme, skupno verjetnostno porazdelitev, vzročno-posledične diagrame in Bayesove mreže. S tem bomo pripravili temelje za verjetnostno sklepanje, kakršnega obravnavata dvanajsto in trinajsto poglavje _Quantifying Uncertainty_ in _Probabilistic reasoning_ v učbeniku {cite}`russell2019aima`.


## Osnovni pojmi verjetnosti in pogojne verjetnosti

Sklepanje v negotovosti običajno temelji na verjetnosti. Pojem verjetnost (angl. _probability_) v praksi navadno interpretiramo kot številsko mero prepričanja, da se dogodek zgodi (oziroma izjava velja, če razmišljamo v okviru sklepanja). Bolj matematično povedano, je verjetnost dogodka predviden (ocenjen, pričakovan) delež poskusov, pri katerih se zgodi opazovani dogodek. Verjetnost je število med $0$ in $1$, pri čemer verjetnost $0$ pripišemo _nemogočemu_ dogodku, $1$ pa _gotovemu_ dogodku. Zapišimo to bolj formalno.

```{prf:definition} Poskus, izid, dogodek in verjetnost dogodka

_Poskus_ je realizacija natančno določenih pogojev, pri katerih opazujemo enega ali več _dogodkov_. Po realizaciji poskusa opazujemo njegov _izid_, iz katerega mora biti jasno, ali se je opazovani dogodek zgodil ali ne.

Množico možnih izidov označimo z $\Omega$. Dogodek je definiran s poljubno podmnožico množice $\Omega$.

_Verjetnost_ je preslikava $P \colon 2^\Omega \to [0, 1]$, ki vsakemu dogodku iz potenčne množice izidov $\Omega$ dodeli njegovo verjetnost.
```

Opazujemo poskus metanja poštenega, simetričnega kovanca. Množica možnih izidov tega poskusa je $\{C, G\}$, cifra in glava. Njena potenčna množica definira štiri možne dogodke, torej $\emptyset$, $\{C\}$, $\{G\}$ in $\{C, G\}$. Prvi dogodek je nemogoč, četrti pa gotov (premisli in pojasni, zakaj), drugi in tretji pa sta dogodka z verjetnostjo $0.5$.

Obravnavaj dogodka "met igralne kocke" in "naključna izbira karte iz običajnega kompleta 52 igralnih kart". V obeh primerih premisli, kaj je množica možnih izidov $\Omega$, in naštej nekaj primerov dogodkov, za katere znaš oceniti njihovo verjetnost.

Definirajmo zdaj pojem slučajne spremenljivke in njene porazdelitve.

```{prf:definition} Slučajna spremenljivka in njena porazdelitev

_Slučajna spremenljivka_ $X$ z vrednostmi v množici $D_X$ je preslikava iz $\Omega$ v $D_X$, ki vsakemu izidu priredi določeno vrednost v $D_X$. Množici $D_X$ rečemo zaloga vrednosti spremenljivke $X$.

_Porazdelitev slučajne spremenljivke_ $X$ pove, s kolikšnimi verjetnostmi zavzame posamezne vrednosti iz zaloge $D_X$. Če je zaloga vrednosti končna množica $D_X = \{ x_1, x_2, \dots, x_k \} $, porazdelitev opisuje shema
$$
    \left(
        \begin{array}{cccc}
            x_1 & x_2 & \dots & x_k \\
            p_1 & p_2 & \dots & p_k
        \end{array}
    \right),
$$
kjer je za vse $i = 1, 2, \dots, k$, $p_i = P(X = x_i)$ , torej verjetnost, da je vrednost spremenljivke $X$ enaka $x_i$.
```

Zapiši verjetnostne sheme za slučajne spremenljivke: $A$, izid metanja poštene igralne kocke in $B$, barva naključno izbrane karte iz običajnega kompleta 52 igralnih kart.

V primeru iz uvodnega razdelka so $G$, $S$ in $V$ slučajne spremenljivke. Vsaka ima zalogo dveh možnih vrednosti, zato jim lahko rečemo boolove slučajne spremenljivke, katerih vrednosti pogosto označimo z `da` in `ne`, lahko pa uporabimo tudi vsebinsko bolj ustrezne oznake, kot sta `je` in `ni`.

Če nas zanima verjetnost, da avtomobil ne vžge, pišemo $P(V=\textit{ne})$. Če nas zanima verjetnost, da v avtomobilu ni goriva, pišemo $P(G=\textit{ni})$.

Pogosto nas ne zanima verjetnost dogodka samega, temveč verjetnost dogodka pod pogojem, da vemo nekaj dodatnega.

```{prf:definition} Pogojna verjetnost

_Pogojna verjetnost_ dogodka $A$ glede na dogodek $B$ (ki ni nemogoč) je verjetnost dogodka $A$, če vemo, da se je zgodil dogodek $B$. Označimo jo z $P(A \mid B)$.
```

Predpostavi, da verjetnost dogodka definiramo kot delež poskusov, pri katerih se ta dogodek zgodi. Premisli, kako v tem primeru izračunamo verjetnost $P(A \mid B)$, če poznamo delež poskusov, kjer sta se hkrati zgodila dogodka $A$ in $B$, torej $P(A \cap B)$, ter delež dogodkov, kjer se zgodi dogodek $B$. Tak premislek nam pomaga izpeljati znano formulo za izračun pogojne verjetnosti:
$$
    P(A \mid B) = \frac{P(A \cap B)}{P(B)}.
$$

Pogojna verjetnost nam pove, kako spremenimo svoje prepričanje o dogodku $A$, ko izvemo, da se je zgodil dogodek $B$.

V primeru avtomobila je tako pomembna verjetnost $P(G=\textit{ni} \mid V=\textit{ne})$, ki nam pove, kakšna je verjetnost, da v avtomobilu ni goriva, če že vemo, da avtomobil ni vžgal. Premisli, zakaj iz poznavanja $P(G=\textit{ni})$ še ne moremo sklepati na $P(G=\textit{ni} \mid V=\textit{ne})$. Kaj je tisto dodatno znanje, ki ga v drugem primeru upoštevamo?

Iz formule za izračun pogojne verjetnosti vemo, da velja
$$
P(G=\textit{ni} \mid V=\textit{ne}) = \frac{P(G=\textit{ni} \land V=\textit{ne})}{P(V=\textit{ne})}.
$$
Za izračun torej potrebujemo verjetnosti sestavljenih dogodkov, ki se nanašajo na vrednosti dveh (in v splošnem več) spremenljivk. V zgornjem primeru je to dogodek, ki ustreza situaciji $G=\textit{ni} \land V=\textit{ne}$.

## Skupna verjetnostna porazdelitev in računanje z verjetnostmi

To nas pripelje do pojma skupne verjetnostne porazdelitve več spremenljivk, ki nam poda verjetnosti vseh možnih kombinacij njihovih vrednosti.

```{prf:definition} Skupna verjetnostna porazdelitev (angl. _joint probability distribution_)

_Skupna verjetnostna porazdelitev_ za množico slučajnih spremenljivk podaja verjetnost vsake popolne prireditve vrednosti tem spremenljivkam.
```

Za tri boolove spremenljivke $G$, $S$ in $V$ ima tabela skupne verjetnostne porazdelitve osem vrstic, saj ima vsaka spremenljivka dve možni vrednosti. Vsaka vrstica določa verjetnost ene možne situacije, na primer:
- $P(G=\textit{ni} \land S=\textit{ne} \land V=\textit{ne})$,
- $P(G=\textit{je} \land S=\textit{da} \land V=\textit{da})$ in
- $P(G=\textit{je} \land S=\textit{ne} \land V=\textit{ne})$.

Ko imamo podano skupno verjetnostno porazdelitev, lahko iz nje izračunamo verjetnost za podane vrednosti podmnožice spremenljivk in s tem tudi poljubne pogojne verjetnosti. Spomnimo se formule iz verjetnosti, ki jo bomo pri tem uporabljali:

Naj bo podana skupna verjetnostna porazdelitev za slučajne spremenljivke $X_1, X_2, \dots, X_n$ z domenami $D_1, D_2, \dots, D_n$. Naj bo $\{i_1, i_2, \dots, i_k\} \subset \{1, 2, \dots, n\}$. Za izračun verjetnosti $P(X_{i_1} = x_{i_1} \land X_{i_2} = x_{i_2} \land \dots \land X_{i_k} = x_{i_k})$ velja enačba:
$$
    P(X_{i_1} = x_{i_1} \land \dots \land X_{i_k} = x_{i_k}) =
    \sum_{x_{j_1} \in D_{j_1}} \dots \sum_{x_{j_{n-k}} \in D_{j_{n-k}}} P(X_{i_1} = x_{i_1} \land \dots \land X_{i_k} = x_{i_k} \land X_{j_1} = x_{j_1} \land \dots \land X_{j_{n-k}} = x_{j_{n-k}}),
$$
kjer je $\{ j_1, j_2, \dots, j_{n-k} \} = \{1, 2, \dots, n\} \setminus \{i_1, i_2, \dots, i_k\}$. Z drugimi besedami, verjetnost za delno prireditev vrednosti spremenljivk dobimo tako, da seštejemo verjetnosti za vsa možna dopolnjevanja prireditve vrednosti spremenljivk do polne prireditve.

Kar nam omogoča, da izračunamo verjetnost, da v avtomobilu ni goriva, če ne vžge takole:
$$
\begin{align*}
    P(G=\textit{ni} \mid V=\textit{ne})
        &= \frac{P(G=\textit{ni} \land V=\textit{ne})}{P(V=\textit{ne})} \\
        &= \frac{\sum_{s \in \{\textit{da}, \textit{ne}\}} P(G=\textit{ni} \land S = s \land V=\textit{ne})}{\sum_{(g, s) \in \{ \textit{je}, \textit{ni} \} \times \{\textit{da}, \textit{ne}\}} P(G=g \land S = s \land V=\textit{ne})}
\end{align*}
$$

Tak način računanja je splošen in konceptualno preprost. Težava pa je v velikosti tabele skupne verjetnostne porazdelitve. Če imamo $m$ spremenljivk in ima vsaka največ $n$ vrednosti, potem velikost tabele skupne porazdelitve naraste kot $O(n^m)$. To pomeni, da postane neposreden zapis tabele hitro prevelik. Ravno zato iščemo bolj strukturirane predstavitve znanja iz tabele skupne porazdelitve. Namesto da bi eksplicitno zapisali vse kombinacije, želimo izkoristiti odvisnosti in neodvisnosti med opazovanimi slučajnimi spremenljivkami.

Na tem mestu je koristno zapisati še eno pomembno formulo, ki jo bomo pogosto uporabljali pri sklepanju iz opažanj nazaj na možne vzroke.

```{prf:definition} Bayesova formula

Naj bosta $A$ in $B$ dogodka z verjetnostma $P(A) > 0$ in $P(B) > 0$. Potem velja
$$
P(A \mid B) = \frac{P(B \mid A) P(A)}{P(B)}.
$$
```

Bayesova formula nam omogoča, da verjetnost vzroka po opažanju posledice izrazimo s pomočjo verjetnosti posledice pri znanem vzroku, začetne verjetnosti vzroka in verjetnosti opažanja. V primeru avtomobila bi jo lahko uporabili za prepis verjetnosti $P(G=\textit{ni} \mid V=\textit{ne})$ v obliko, ki vključuje $P(V=\textit{ne} \mid G=\textit{ni})$, začetno verjetnost $P(G=\textit{ni})$ in verjetnost opažanja $P(V=\textit{ne})$.

```{prf:definition} Neodvisna dogodka

Dogodka $A$ in $B$ sta _neodvisna_, če velja $P(A \cap B) = P(A) P(B)$.
```

To pomeni, da informacija o tem, ali se je zgodil dogodek $B$, ne spremeni verjetnosti dogodka $A$ (in obratno).

Premisli, ali sta pri metu igralne kocke dogodka "liho število pik" in "sodo število pik" neodvisna. Pojasni, zakaj. Premisli zdaj o hkratnem metanju dveh igralnih kock. Ali sta dogodka "na prvi kocki je liho število točk" in "na drugi kocki je sodo število točk" neodvisna? Pojasni, zakaj.

```{prf:definition} Neodvisni slučajni spremenljivki

Slučajni spremenljivki $X$ in $Y$ sta _neodvisni_, če za vse vrednosti $x \in D_X$ in $y \in D_Y$ velja $P(X=x \land Y=y) = P(X=x) P(Y=y)$.
```

Premisli kateri par spremenljivk iz primera v uvodnem razdelku sta neodvisni.

```{prf:definition} Pogojna neodvisnost

Slučajni spremenljivki $X$ in $Y$ sta _pogojno neodvisni_ glede na slučajno spremenljivko $Z$, če za vse vrednosti $x \in D_X$, $y \in D_Y$ in $z \in D_Z$, za katere velja $P(Z=z) > 0$, velja
$$
P(X=x \land Y=y \mid Z=z) = P(X=x \mid Z=z) P(Y=y \mid Z=z).
$$
To označimo z $X \perp Y \mid Z$.
```

Pogojna neodvisnost je osrednja ideja Bayesovih mrež. Prav zaradi nje lahko skupno verjetnostno porazdelitev zapišemo mnogo bolj kompaktno, kot bi jo lahko zapisali neposredno s celotno tabelo. Preden spoznamo Bayesove mreže, bomo obravnavali pojem vzročno-posledičnih diagramov.

## Vzročno-posledični diagrami in Bayesove mreže

Pri verjetnostnem modeliranju pogosto ne opazujemo le številskih verjetnosti, temveč tudi strukturo vplivov med spremenljivkami. Intuitivno nas zanima, katera spremenljivka vpliva na katero in kakšna je smer tega vpliva. V primeru vžiga avtomobila lahko rečemo, da na spremenljivko `VžigAvtomobila` vplivata `Gorivo` in `ČisteSvečke`. To lahko prikažemo z usmerjenim diagramom, v katerem puščice kažejo od vzrokov proti posledici.

Tak diagram ni še formalna Bayesova mreža, vendar nam že daje pomembno intuitivno informacijo o strukturi problema. Poleg tega nam omogoča razmislek o različnih vrstah povezav med spremenljivkami. Ločimo tri osnovne oblike povezav:

1. _serijske povezave_ oblike $A \rightarrow B \rightarrow C$,
1. _divergentne povezave_ oblike $A \rightarrow B$ in $A \rightarrow C$,
1. _konvergentne povezave_ oblike $B \rightarrow A \leftarrow C$.

Pri serijski povezavi lahko informacija teče po verigi. Če opazimo posledico $C$, lahko sklepamo nazaj proti $B$ in posredno tudi proti $A$. Naj bo $A$ dogodek `Vlom`, $B$ dogodek `Alarm`, $C$ pa dogodek `Klic varnostne službe`. Če izvemo, da se je zgodil klic varnostne službe, lahko sklepamo, da je bil verjetno sprožen alarm, od tod pa naprej, da je verjetno prišlo do vloma. Podobno, če vemo, da je prišlo do vloma, lahko sklepamo, da je verjetno zazvonil alarm, in posledično tudi, da je verjeten klic varnostne službe. V taki verigi se torej vpliv prenaša po poti od vzroka k posledici in nazaj.

Pri divergentni povezavi skupni vzrok vpliva na več posledic. Če poznamo vzrok, lahko sklepamo o obeh posledicah; če pa poznamo eno posledico, lahko pogosto sklepamo na vzrok in nato naprej na drugo posledico. Naj bo $A$ spremenljivka `Spol`, $B$ spremenljivka `Dolžina las`, $C$ pa `Višina`. Če poznamo vzrok $A$, na primer da gre za žensko osebo, lahko iz tega sklepamo, da so dolgi lasje bolj verjetni in da je tudi manjša telesna višina bolj verjetna. Če pa opazimo samo dolge lase, lahko to poveča verjetnost, da gre za žensko osebo, iz tega pa lahko posredno sklepamo tudi na bolj verjetno manjšo višino. Divergentna povezava torej omogoča sklepanje od skupnega vzroka proti več posledicam in posredno tudi med posledicami prek skupnega vzroka.

Posebej zanimiva je konvergentna povezava, kjer imata dva vzroka skupno posledico. V primeru avtomobila sta `Gorivo` in `ČisteSvečke` dva možna vzroka za `VžigAvtomobila`. Če ne vemo nič o posledici, sta vzroka med seboj lahko neodvisna. Ko pa izvemo, da avtomobil ni vžgal, informacija o enem vzroku vpliva na oceno drugega. Temu pojavu pogosto rečemo _pojasnjevanje_ (angl. _explaining away_). Tako na primer, če vemo, da avtomobil ni vžgal, in dodatno ugotovimo, da v avtomobilu ni goriva, se zmanjša potreba po razlagi s pomočjo umazanih svečk. Informacija o enem vzroku torej spremeni verjetnost drugega vzroka.

Vzročno-posledični diagrami so pomembni zato, ker nam omogočijo bolj kompakten zapis skupne verjetnostne porazdelitve. To idejo formaliziramo z Bayesovimi mrežami.

Premisli o tem, kaj bi bili nadaljnji primeri zaporednih, divergentnih in konvergentnih povezav.

Bayesove mreže so formalna predstavitev verjetnostnega znanja, ki združi dve vrsti informacije:

- strukturo vplivov med slučajnimi spremenljivkami in
- verjetnostne porazdelitve, ki te vplive kvantificirajo.

```{prf:definition} Bayesova mreža

_Bayesova mreža_ je aciklični usmerjeni graf, v katerem so vozlišča slučajne spremenljivke, povezave pa predstavljajo neposredne vplive med njimi. Vsakemu vozlišču $X$ je pripisana verjetnostna porazdelitev $P(X \mid Y_1, Y_2, \dots, Y_m)$, kjer so $Y_1, Y_2, \dots, Y_m$ starši vozlišča $X$.
```

Če vozlišče nima staršev, mu pripada navadna verjetnostna porazdelitev $P(X)$. Če starše ima, mu pripada pogojna verjetnostna porazdelitev glede na te starše, kot jo določa zgornja definicija.

Za serijsko povezavo $A \rightarrow B \rightarrow C$ tako potrebujemo:

- $P(A)$,
- $P(B \mid A)$ in
- $P(C \mid B)$.

Pri tem ni treba posebej zapisati celotne skupne porazdelitve $P(A,B,C)$, saj jo lahko rekonstruiramo iz teh lokalnih porazdelitev s pomočjo verižnega pravila.

```{prf:definition} Faktorizacija skupne porazdelitve v Bayesovi mreži

Če Bayesova mreža vsebuje vozlišča $X_1, X_2, \dots, X_n$, potem se skupna verjetnostna porazdelitev zapiše kot produkt lokalnih porazdelitev:
$$
P(X_1, X_2, \dots, X_n) = \prod_{i=1}^{n} P(X_i \mid Pa(X_i)).
$$
```

To je ključna prednost Bayesovih mrež. Namesto eksponentno velike tabele pogosto zadostuje bistveno manjši nabor lokalnih tabel. Če ima posamezno vozlišče malo staršev, je opis mreže mnogo bolj kompakten od popolne tabele skupne verjetnostne porazdelitve.

Pri gradnji Bayesove mreže moramo določiti:

1. _strukturo_ mreže, torej katera vozlišča so povezana in v katero smer,
1. _parametre_ mreže, torej verjetnostne porazdelitve v posameznih vozliščih.

Strukturo običajno določimo na osnovi znanja o domeni. Parametre lahko podamo ročno, jih ocenimo iz podatkov ali jih izpeljemo z modeliranjem opazovanega sistema.

Premisli, katere lokalne verjetnostne porazdelitve bi morali podati za Bayesovo mrežo z vozlišči `G`, `S` in `V`, kjer velja $G \rightarrow V$ in $S \rightarrow V$.

## Sklepanje v Bayesovih mrežah

Glavni namen Bayesovih mrež ni le kompakten zapis verjetnosti, temveč predvsem učinkovito verjetnostno sklepanje. Pri sklepanju nas zanimajo vprašanja tipa:

- kakšna je verjetnost vzroka, če opazimo posledico,
- kako se spremeni verjetnost neke spremenljivke, če opazimo drugo,
- katera razlaga opazovanega dogodka je verjetnejša.

Razmislimo o klasičnem primeru s petimi boolovimi spremenljivkami: vlom $V$, strela $S$, tipalo $T$, alarm $A$ in klic $K$. Intuitivni model je naslednji: vlom ali strela lahko sprožita tipalo; sproženo tipalo lahko aktivira alarm; sproženo tipalo lahko povzroči tudi klic varnostne službe. Strukturo te mreže prikazuje {ref}`fig-bm-alarm`. V Bayesovi mreži s slike to pomeni, da lahko iz lokalnih porazdelitev $P(V)$, $P(S)$, $P(T \mid V, S)$, $P(A \mid T)$ in $P(K \mid T)$ izračunamo na primer $P(V \mid A)$, torej verjetnost vloma ob opaženem alarmu.

```{figure} ../materiali/bm-alarm.png
---
name: fig-bm-alarm
---
Bayesova mreža za primer protivlomnega alarma.
```

Pri takem računu najprej ločimo tri vrste spremenljivk. Poizvedba je $V$, ker nas zanima verjetnost vloma. Dokaz je $A$, ker predpostavimo, da je alarm opažen. Preostale spremenljivke $S$, $T$ in $K$ so skrite spremenljivke, po katerih moramo v splošnem seštevati.

Prvi korak je zapis po definiciji pogojne verjetnosti:
$$
P(V \mid A) = \frac{P(V, A)}{P(A)}.
$$

V števcu nato izrazimo skupno verjetnost s seštevanjem po skritih spremenljivkah:
$$
P(V, A) = \sum_{s,t,k} P(V, s, t, A, k).
$$
Ker mreža razcepi skupno porazdelitev na produkt lokalnih porazdelitev, dobimo:
$$
P(V, A) = \sum_{s,t,k} P(V) P(s) P(t \mid V, s) P(A \mid t) P(k \mid t).
$$

Naslednji korak je poenostavitev. Spremenljivka $K$ ne nastopa v poizvedbi, zato lahko vsoto po njej izvedemo takoj. Za vsak fiksni $t$ velja
$$
\sum_k P(k \mid t) = 1,
$$
zato se člen za $K$ izniči. Števec se zato poenostavi v izraz
$$
P(V, A) = P(V) \sum_{s,t} P(s) P(t \mid V, s) P(A \mid t).
$$

Podobno razvijemo še imenovalec. Ker je $A$ edini dokaz, moramo pri $P(A)$ sešteti po vseh ostalih spremenljivkah:
$$
P(A) = \sum_{v,s,t,k} P(v, s, t, A, k).
$$
Po enaki faktorizaciji in isti poenostavitvi po $K$ dobimo:
$$
P(A) = \sum_{v,s,t} P(v) P(s) P(t \mid v, s) P(A \mid t).
$$

Če ta izraza vstavimo v začetno formulo, dobimo končni zapis:
$$
P(V \mid A) = \frac{P(V) \sum_{s,t} P(s) P(t \mid V, s) P(A \mid t)}{\sum_{v,s,t} P(v) P(s) P(t \mid v, s) P(A \mid t)}.
$$
To še ni numerični rezultat, je pa popoln načrt izračuna. Če bi poznali konkretne tabele verjetnosti za podano Bayesovo mrežo, bi v ta izraz le vstavili ustrezne številčne vrednosti in izračunali rezultat.

Pomembna strategija pri takem sklepanju je torej, da najprej jasno ločimo poizvedbo, dokaze in skrite spremenljivke. Pri ročnem računanju je koristno najprej zapisati cilj v obliki $P(Q \mid E)$, nato pa načrtno razvijati števec in imenovalec, dokler ne pridemo do verjetnosti, ki jih podana Bayesova mreža neposredno podaja.

Isto idejo lahko zapišemo tudi v obliki preproste psevdokode za sklepanje s popolnim naštevanjem, ki jo podaja algoritem {ref}`alg-enumeration-ask`. Funkcija `NAŠTEVANJE_POIZVEDBA` prejme poizvedbo, to je slučajno spremenljivko `Q`, opažanja oziroma dokaze `E` in Bayesovo mrežo `B`. Za vsako možno vrednost spremenljivke `Q` izračuna nenormirano verjetnost, nato pa rezultat normira.

V zapisu algoritma `Pa(X)` označuje množico staršev vozlišča `X` v Bayesovi mreži. Če vozlišče ustreza spremenljivki `A` iz mreže za alarm na {ref}`fig-bm-alarm`, potem je `Pa(A) = {T}`. Če je vozlišče `T`, pa je `Pa(T) = {V, S}`. Funkcija `SPREMENLJIVKE(B)` vrne urejen seznam vseh spremenljivk v mreži `B`. Za primer iz tega razdelka bi funkcijo poklicali z ukazom `NAŠTEVANJE_POIZVEDBA(V, {A=da}, B)`, kjer je `B` Bayesova mreža za alarm. Tak klic pomeni: izračunaj verjetnostno porazdelitev za spremenljivko `V`, če vemo, da se je sprožil alarm, torej če velja `A = da`.

V prvem delu funkcija `NAŠTEVANJE_POIZVEDBA` preizkusi vse možne vrednosti `q` poizvedbe `Q` in za vsako pokliče pomožno funkcijo `NAŠTEJ_VSE` tako, da dokazom `E` doda opažanje `Q = q` (vrstici 9-10). Tako pripravi slovar `f`, ki po normalizaciji predstavlja verjetnostno porazdelitev spremenljivke `Q` (vrstica 11). Za boolovo spremenljivko `V` iz primera to pomeni, da algoritem izračuna verjetnosti za `V = da` in `V = ne`.

```{code-block} text
:caption: Osnovni algoritem sklepanja iz Bayesove mreže s popolnim naštevanjem
:name: alg-enumeration-ask
:linenos:

VHOD:
  Q je poizvedba (spremenljivka)
  E je množica opažanj (dokazov)
  B je Bayesova mreža

IZHOD: verjetnostna porazdelitev za Q

NAŠTEVANJE_POIZVEDBA(Q, E, B)
  for vrednost q spremenljivke Q
    f(q) ← NAŠTEJ_VSE(SPREMENLJIVKE(B), E \cup {Q = q})
  return NORMALIZIRAJ(f)


# prvi argument je seznam spremenljivk, E je množica dokazov
NAŠTEJ_VSE([X_1, ..., X_n], E)
  if seznam je prazen
    return 1
  X ← prvi element seznama
  if X = x je opažanje iz E
    return P(x | Pa(X)) * NAŠTEJ_VSE(preostanek seznama, E)
  else
    vsota ← 0
    for vrednost x spremenljivke X
      vsota ← vsota + P(x | Pa(X)) * NAŠTEJ_VSE(preostanek seznama, E \cup {X = x})
    return vsota
```

Funkcija `NAŠTEJ_VSE` rekurzivno obdeluje spremenljivke po vrsti. Če je vrednost trenutne spremenljivke že določena z opažanjem, ustrezno lokalno verjetnost pomnoži z rezultatom za preostanek seznama (vrstica 20). Če spremenljivka ni določena z opažanjem, gre za skrito spremenljivko; v tem primeru sešteje prispevke po vseh možnih vrednostih te spremenljivke (vrsitce 22-25). Robni pogoj rekurzije nastopi, ko je seznam spremenljivk prazen. Takrat algoritem vrne 1 (vrstici 16-17).

Predstavljeni algoritem sledi ročnemu izračunu iz zgornjega primera. Če je vrednost neke spremenljivke že določena z dokazi ali s trenutno obravnavano vrednostjo poizvedbe, njen prispevek le pomnožimo. Če vrednost ni določena, moramo po vseh možnostih sešteti. Na koncu dobljene nenormirane vrednosti pretvorimo v pravo porazdelitev.

Pred dejanskim računanjem pogosto preverimo tudi, kateri deli mreže s {ref}`fig-bm-alarm` so glede na poizvedbo in dokaze sploh relevantni. S pomočjo separabilnosti oziroma d-ločevanja lahko iz obravnave izločimo vozlišča, ki na poizvedbo ne morejo vplivati, in tako zmanjšamo obseg izračuna. Separabilnost zato ni samostojen algoritem sklepanja, temveč pomembna strategija za poenostavitev problema.

Premisli, kako bi se zgornji izraz in potek algoritma spremenila, če bi poleg alarma opazili še klic varnostne službe in bi torej računali $P(V \mid A, K)$.

# Naloga za bonus točke

1. (5 točk, rok oddaje 14. april) V Pythonovem paketu [`pgmpy`](https://pgmpy.org/) implementiraj Bayesovo mrežo za primer alarma iz zadnjega razdelka, prikazano na {ref}`fig-bm-alarm`. Za vsako vozlišče podaj ustrezno tabelo verjetnosti. Pri tem uporabi smiselne vrednosti po lastni izbiri in pri tem pazi, da bodo porazdelitve veljavne. Z implementirano mrežo izračunaj verjetnosti $P(V \mid A=\textit{da})$ in $P(V \mid A=\textit{da}, K=\textit{ne})$. Na kratko komentiraj rezultat: ali opazovanje $K=\textit{ne}$ poveča ali zmanjša verjetnost vloma glede na primer, ko vemo zgolj, da je alarm sprožen?<br/><br/>Namigi za uporabo `pgmpy`: razred [`DiscreteBayesianNetwork`](https://pgmpy.org/models/bayesiannetwork.html) omogoča ustvarjanje strukture Bayesove mreže, razred [`TabularCPD`](https://pgmpy.org/factors/discrete.html) podajanje tabel verjetnosti, metoda [`check_model`](https://pgmpy.org/models/bayesiannetwork.html) preverjanje pravilnosti mreže in tabel, razred [`VariableElimination`](https://pgmpy.org/exact_infer/ve.html) in njegova metoda [`query`](https://pgmpy.org/exact_infer/ve.html) pa omogočata sklepanje. 
<br/><br/>
